# 22 · Tensor parallelism and all-reduce
Level L4 · Part 3 · core · ~20 min · builds on 19

## Goal
By the end of this lesson you can show how tensor parallelism splits the heads and the MLP of the 70B model over 8 GPUs, count the all-reduces in one forward pass, and compute what they cost over PCIe and over NVLink. You can also say which cost dominates in decode and which in prefill.

## Why it matters
- **Without it:** you set `--tensor-parallel-size 8` on an 8-GPU box. The model loads, the first token is slow, and you blame the model. You never count the 160 collective calls that every forward pass makes.
- **With it:** the config numbers and one link speed give you the communication cost of a prompt before you rent the box. This lesson uses the 1,600-token Brightside prompt.
- **When you don't need it:** a model that fits on one GPU with room for the cache needs no split. Every split adds all-reduces, so split only when the weights or the cache do not fit (lesson 19).

## The idea
Tensor parallelism cuts each large matrix into pieces, one piece per GPU, in two ways. A **column cut** gives each GPU a slice of the output, so it needs nothing from the others yet. A **row cut** gives each GPU a partial sum over its slice of the input, so the partial sums must be added together. That adding step is the **all-reduce**. A transformer layer has two row cuts (after attention and after the MLP), so there are two all-reduces per layer: 160 per forward pass for 80 layers.

Think of eight accountants who each add up one part of a ledger. Before anyone can report the total, they pass their subtotals around the table.

## Picture
One layer on TP=8. Every GPU runs the same steps on its own slice, and the two all-reduces join them:
```mermaid
flowchart LR
  X["token activations"] --> QKV["QKV: column cut by heads"]
  QKV --> AT["attention: 8 of 64 query heads per GPU"]
  AT --> O["output projection: row cut"]
  O --> AR1["all-reduce 1"]
  AR1 --> UP["MLP up and gate: column cut"]
  UP --> DN["MLP down: row cut"]
  DN --> AR2["all-reduce 2"]
  AR2 --> Y["layer output, same on every GPU"]
```

## Step by step
### Step 1: the numbers
These are copies of the public Llama 3.1 70B `config.json` values. Meta's repository is gated, so the values come from an ungated mirror of the same file. Lesson 19 used the first four numbers; the MLP width is new here. Nothing is downloaded.

In [1]:
import shutil

GIB = 2**30
KIB = 2**10
L4_USABLE_GIB = 24e9 * 0.92 / GIB       # one L4 for vLLM (lesson 19): 20.56 GiB

LAYERS, Q_HEADS, KV_HEADS = 80, 64, 8
HIDDEN, MLP_WIDTH = 8192, 28672
PARAMS_70B = 70_553_706_496
BYTES = 2                                # FP16
ACT_BYTES_PER_TOKEN = HIDDEN * BYTES     # one token's activation vector: 16 KiB
ALLREDUCES_PER_PASS = LAYERS * 2         # after attention, after the MLP

print(f"activation per token: {ACT_BYTES_PER_TOKEN // KIB} KiB")
print(f"all-reduces per forward pass: {ALLREDUCES_PER_PASS}")

activation per token: 16 KiB
all-reduces per forward pass: 160


**What you should see:** 16 KiB per token and 160 all-reduces per forward pass. The 160 does not change with the degree: the count is set by the layers.

### Step 2: what one GPU holds
Attention splits by heads: each GPU computes its own query heads and its own KV heads. The MLP splits its width. Both splits must divide evenly, so the degree must divide 64 and 8. The MLP width, 28672, divides by all four degrees here.

In [2]:
def split(tp):
    """What one GPU holds when the model is split tp ways."""
    if Q_HEADS % tp or KV_HEADS % tp:
        raise ValueError(f"TP={tp} must divide {Q_HEADS} query and {KV_HEADS} KV heads")
    return {
        "query heads": Q_HEADS // tp,
        "KV heads": KV_HEADS // tp,
        "MLP columns": MLP_WIDTH // tp,
        "FP16 weights (GiB)": round(PARAMS_70B * BYTES / tp / GIB, 1),
    }

for tp in (1, 2, 4, 8):
    print(f"TP={tp}", split(tp))

TP=1 {'query heads': 64, 'KV heads': 8, 'MLP columns': 28672, 'FP16 weights (GiB)': 131.4}
TP=2 {'query heads': 32, 'KV heads': 4, 'MLP columns': 14336, 'FP16 weights (GiB)': 65.7}
TP=4 {'query heads': 16, 'KV heads': 2, 'MLP columns': 7168, 'FP16 weights (GiB)': 32.9}
TP=8 {'query heads': 8, 'KV heads': 1, 'MLP columns': 3584, 'FP16 weights (GiB)': 16.4}


**What you should see:** at TP=8 each GPU holds 8 query heads, 1 KV head, 3,584 MLP columns and 16.4 GiB of weights. The KV cache follows the KV heads, so each GPU also keeps 1/8 of the cache. The 8 KV heads also cap the split: this lesson goes no higher than TP=8.

### Step 3: the cost of one all-reduce
A ring all-reduce of `M` bytes over `N` GPUs moves `2 × (N − 1) / N × M` bytes through each GPU. Each GPU passes its data around the ring twice: once to add the partial sums, once to share the total. `M` is the tokens in the pass times 16 KiB. Time is bytes divided by link speed, plus a fixed cost per call.

**PCIe** is the ordinary slot link between a GPU and the host. **NVLink** is NVIDIA's faster direct GPU-to-GPU link. An L4 has PCIe Gen4 x16 at about 32 GB/s each way. An H100 has NVLink at 450 GB/s each way. The **fixed cost per call is an assumption** (20 µs), not a measurement. Measure it with **NCCL**, NVIDIA's communication library that does the all-reduce, before you trust the totals.

In [3]:
PCIE_GBS = 32e9       # PCIe Gen4 x16, one direction (theoretical peak)
NVLINK_GBS = 450e9    # H100 NVLink, one direction (NVIDIA: 900 GB/s both ways)
FIXED_US = 20         # ASSUMPTION: fixed cost of one all-reduce call

def allreduce_ms(tokens, tp, link_bytes_per_s):
    """One ring all-reduce of the activations of `tokens` tokens."""
    message = tokens * ACT_BYTES_PER_TOKEN
    moved = 2 * (tp - 1) / tp * message          # bytes each GPU sends and receives
    return (moved / link_bytes_per_s + FIXED_US * 1e-6) * 1e3

def pass_ms(tokens, tp, link_bytes_per_s):
    """All-reduce time of one whole forward pass (160 calls)."""
    return ALLREDUCES_PER_PASS * allreduce_ms(tokens, tp, link_bytes_per_s)

print(f"one 1,600-token all-reduce, TP=8, PCIe: {allreduce_ms(1600, 8, PCIE_GBS):.2f} ms")

one 1,600-token all-reduce, TP=8, PCIe: 1.45 ms


### Step 4: decode and prefill on the same eight GPUs
A decode step handles one new token. A prefill handles the 1,600-token prompt in one pass. Both run the same 160 calls, but the messages differ in size by 1,600 times.

In [4]:
for name, tokens in (("decode, 1 token", 1), ("prefill, 1,600 tokens", 1600)):
    pcie = pass_ms(tokens, 8, PCIE_GBS)
    nvl = pass_ms(tokens, 8, NVLINK_GBS)
    print(f"{name:22} PCIe {pcie:7.1f} ms   NVLink {nvl:6.1f} ms   (all-reduce time, TP=8)")

decode, 1 token        PCIe     3.3 ms   NVLink    3.2 ms   (all-reduce time, TP=8)
prefill, 1,600 tokens  PCIe   232.6 ms   NVLink   19.5 ms   (all-reduce time, TP=8)


**What you should see:** in decode, PCIe and NVLink are almost equal, about 3 ms, because under the 20 µs assumption the fixed cost of 160 calls is most of the time. In prefill, PCIe spends about 230 ms on communication and NVLink about 20 ms. The bytes dominate here. NVLink is about 12 times less all-reduce time in total (the link is 14 times faster; the fixed cost keeps the gap smaller). These are lower bounds: real links reach less than their peak.

### Step 5: the console walkthrough (on a GPU host)
On a laptop you read these steps. On a GPU host with eight GPUs you run them. Each step says what you should see.

**1. Read the link map.**
```bash
nvidia-smi topo -m
```
You should see a matrix of GPU pairs. `NV#` means the pair has NVLink. `PHB`, `PIX`, `NODE` or `SYS` mean the traffic crosses PCIe or the host. An L4 box shows no `NV#` entries.


**2. Find the flag on your version.**
```bash
vllm serve --help=all | grep -i "tensor-parallel"
```
You should see `--tensor-parallel-size` (short form `-tp`) with default 1.


**3. Start the model on eight GPUs.**
```bash
vllm serve meta-llama/Llama-3.1-70B-Instruct --tensor-parallel-size 8 --gpu-memory-utilization 0.92
```
You should see one worker per GPU in the log. The `/health` endpoint answers only after every worker is ready.


**4. Name the transport between GPUs.**
```bash
NCCL_DEBUG=TRACE vllm serve meta-llama/Llama-3.1-70B-Instruct --tensor-parallel-size 8 > nccl.log 2>&1
grep -E "via (P2P|SHM|NET)" nccl.log | head
```
You should see `P2P` for a direct GPU-to-GPU path, which is PCIe or NVLink. `SHM` means the data goes through host memory, which is slower. Across machines, `NET/IB` is InfiniBand and `NET/Socket` is plain TCP, which is slow for all-reduce. The log is long, so write it to a file and grep it.

In [5]:
has_gpu = shutil.which("nvidia-smi") is not None
print(f"NVIDIA driver on this machine: {has_gpu}")
print("On a laptop, steps 1 to 4 are read-only. The arithmetic above runs anywhere.")

NVIDIA driver on this machine: False
On a laptop, steps 1 to 4 are read-only. The arithmetic above runs anywhere.


### What tensor parallelism does not solve
- **Throughput.** TP makes one copy of the model fit and run faster. More copies, called replicas, give more requests per second (taught later, in lesson 29).
- **Slow links.** TP across machines puts every all-reduce on the network. Keep TP inside one node. Split across nodes with pipeline parallelism (each node holds a different group of layers; lesson 34 covers the rule).

## At work
- **vLLM issue tracker:** teams hit tensor-parallel settings in bug reports, so check the flag's help text for your version (step 2 above). → lesson 23 sizes the TP degree for a 70B model.
- **Meta and PyTorch (Sep 2025)** ran disaggregated serving (prefill and decode on separate machines) on 8-H100 hosts. Over TCP, TTFT (time to first token) regressed sharply under heavy load: moving data between machines was the bottleneck. → lesson 34.
- **vLLM documentation:** the tensor-parallel size is the number of GPUs in one node; across nodes, pipeline parallelism is added. → lesson 34.

Sources: Meta and PyTorch, "Disaggregated Inference at Scale with PyTorch & vLLM" (2025-09-12): https://pytorch.org/blog/disaggregated-inference-at-scale-with-pytorch-vllm/ · vLLM issues: https://github.com/vllm-project/vllm/issues · vLLM docs: https://docs.vllm.ai/

## Common mistakes
**1. Choosing a degree that does not divide the heads.** Three GPUs look convenient, but 64 query heads do not split into three equal parts. The split refuses this.

In [6]:
split(3)   # TP=3: 64 query heads cannot be split in three

ValueError: TP=3 must divide 64 query and 8 KV heads

The fix is a degree that divides 64 and 8: 1, 2, 4 or 8.

**2. Tuning the bytes when the calls dominate.** A decode step over PCIe is slow because of the 160 calls, not the bytes (under the 20 µs assumption). A faster link helps prefill much more than decode. Larger batches and fewer calls per token help decode. Measure the fixed cost on your box before you choose.

## Try it
Your team has four L4s on PCIe. Use `pass_ms` to compute the all-reduce time of one 1,600-token prefill at TP=4. Then use `split(4)` to check whether FP16 70B fits on those cards.

In [7]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [8]:
ms = pass_ms(1600, 4, PCIE_GBS)
print(f"TP=4, 1,600-token prefill over PCIe: {ms:.1f} ms of all-reduce")
w = split(4)["FP16 weights (GiB)"]
print(f"FP16 weights per L4 at TP=4: {w} GiB, usable {L4_USABLE_GIB:.2f} GiB, fits: {w < L4_USABLE_GIB}")

TP=4, 1,600-token prefill over PCIe: 199.8 ms of all-reduce
FP16 weights per L4 at TP=4: 32.9 GiB, usable 20.56 GiB, fits: False


The check prints `False`: FP16 70B does not fit on four L4s. You would need TP=8 on L4s, or a smaller or quantized model (lessons 20 and 23).

## Recap
- Tensor parallelism cuts each matrix two ways. A row cut needs an all-reduce to add the partial sums.
- A 70B model has 80 layers and two all-reduces each: 160 calls per forward pass at any degree.
- At TP=8 each GPU holds 8 query heads, 1 KV head and 3,584 MLP columns.
- Over PCIe, bytes dominate prefill (about 230 ms for 1,600 tokens), and the call count dominates decode (under the 20 µs assumption). NVLink removes most of the prefill cost.

**Interview:** "What does the all-reduce cost in tensor parallelism, and why does it hurt more on PCIe?" Strong answer: two all-reduces per layer, 160 per pass for 70B. Each moves 2(N−1)/N of the activation bytes. A 1,600-token prefill moves about 25 MiB per call: over a millisecond per call on PCIe at about 32 GB/s, about 12 times less in total on NVLink (14× on the bytes alone). Under a 20 µs per-call assumption, decode is limited by the call count.